# feat_crn2 — Query + Demographic **SCORING / INFERENCE** (churn pre-closure model)

Strictly a **scoring** notebook: builds the exact features the trained query model
(`model_cust_query.pkl`) expects and outputs **predictions**. No labels, no 90-day outcome,
no `preclosure_flag`, no train/test split, no model fitting.

Reproduces the Jan-26 OOT pattern (`2_6_oot_data_prep` + `2_9_oot_validation`):
1. full case features via `create_preclosure_features_v4` (verbatim), as of `SNAPSHOT_DATE`
2. bridge loan_number -> opp_id -> lead_id; pull demographics (`net_monthly_sal`, `age_bucket`, `mailing_state`)
3. select **`model.feature_names`** and `predict`; save features + scores

`SNAPSHOT_DATE` = last day of month before `COHORT` (mar_26 -> 2026-02-28). Case cutoff is
strict `createddate < SNAPSHOT_DATE` (fatal leakage guard). The bureau-vintage check is advisory
(warn-only) because the query model never consumes the CRIF scrub.

## 0a. Install pinned dependencies (run once)
Installs the exact runtime into **this** kernel. Comment the cell out after the first successful run. See `requirements.txt` for the pins and the rationale (xgboost 1.7.4, numpy<2.0).

In [ ]:
# Install the production-matched runtime into THIS kernel (see requirements.txt).
# Better: create a dedicated env first  ->  conda create -n churn python=3.12 -y
# Comment this out after the first successful install.
%pip install -q -r requirements.txt


## 0. Configuration — the only cell you edit

In [ ]:
import os
from datetime import datetime
from dateutil.relativedelta import relativedelta

# ============================ EDIT ME ============================
COHORT          = "sep_25"     # "<mon>_<yy>": same token as feat_crn
BUILD_FROM_ROOT = False        # True = pull opp/case/demo from Redshift; False = read existing artifacts
# =================================================================

DRIVE_ROOT = "/Users/khushi.goyal/Library/CloudStorage/GoogleDrive-khushi.goyal@dmifinance.in/Shared drives/DS Team Data/09_Handover/Sanoj/Customer Churn Model"
MODEL_QUERY_PKL = os.path.join(DRIVE_ROOT, "code_3_feature_reduction_and_modelling", "model_cust_query.pkl")
ADDITIONAL_FEATS = ['net_monthly_sal', 'age_bucket', 'mailing_state']   # demographic features (from 2_6 OOT prep)

KNOWN = {
    "jul_25": dict(case="dim_case_data_july25_ids.csv", opploan="opp_id_with_loan_number_jul25_ids.csv", scrub="DMI_CDR_file_july_2lac_PII_CRIF_13-08-2026_output"),
    "sep_25": dict(case="dim_case_data_sep25_ids.csv",  opploan="opp_id_with_loan_number_sep25_ids.csv", scrub="DMI_CDR_file_sep_1_8lac_PII_CRIF_13-08-2026_output"),
    "nov_25": dict(case="dim_case_data_nov25_ids.csv",  opploan="opp_id_with_loan_number_nov25_ids.csv", scrub="DMI_CDR_file_nov_1_6lac_PII_CRIF_13-08-2026_output"),
    "jan_26": dict(case="dim_case_data_jan26_ids.csv",  opploan="opp_id_with_loan_number_jan26_ids.csv", scrub="DMI_CDR_file_jan_1_4lac_PII_CRIF_13-08-2026_output"),
}
_MON = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}
_ms, _yy = COHORT.split("_")
cohort_month  = datetime(2000 + int(_yy), _MON[_ms[:3].lower()], 1)
YEAR_MONTH    = cohort_month.strftime("%Y-%m")
snapshot      = cohort_month - relativedelta(days=1)
RETRO_DATE    = snapshot.strftime("%d-%m-%Y")
SNAPSHOT_DATE = snapshot.strftime("%Y-%m-%d")                 # case leakage cutoff (createddate < this)

# ---- base-selection params (shared root: builds the SAME population as feat_crn, no Drive dependency) ----
BASE_HISTORY_MONTHS    = 8
BASE_DATE_START        = (cohort_month - relativedelta(months=BASE_HISTORY_MONTHS)).strftime("%Y-%m-%d")
cohort_next_month      = cohort_month + relativedelta(months=1)
BASE_DATE_END          = (cohort_next_month - relativedelta(days=1)).strftime("%Y-%m-%d")  # last day of cohort month (include cohort-month EMIs)
BASE_MIN_DUE_DATE      = BASE_DATE_START
BASE_REF_DATE          = cohort_next_month.strftime("%Y-%m-%d")  # base-selection fifo as-of (first day AFTER cohort month); NOT the snapshot
MIN_REMAINING_PAYMENTS = 2
PARTNER_LIST = ['GooglePay','X-Sell','DMID2C','Samsung','AirtelFinance','SuperMoney','Oppo',
                'Finnable','Airtel','Paisabazaar','MoneyView','Billcut','Niro']

_k = KNOWN.get(COHORT, {})
CFG = dict(year_month=YEAR_MONTH,
           case=_k.get("case", f"dim_case_data_{COHORT}_ids.csv"),
           opploan=_k.get("opploan", f"opp_id_with_loan_number_{COHORT}_ids.csv"),
           scrub=_k.get("scrub", f"DMI_CDR_file_{COHORT}_PII_CRIF_output"))

QF_DIR      = os.path.join(DRIVE_ROOT, "code_2_feature_creation", "query_and_flag_creation")
QF_DATA_DIR = os.path.join(QF_DIR, "data")
CATEGORIES_JSON   = os.path.join(QF_DIR, "categories.json")
DRIVE_MAPPING_CSV = os.path.join(DRIVE_ROOT, "data", "final_ids_opp_lead_mapping.csv")

OUT_DIR = os.path.join(os.getcwd(), "output", COHORT); RAW_DIR = os.path.join(OUT_DIR, "raw")
os.makedirs(RAW_DIR, exist_ok=True)

if BUILD_FROM_ROOT:
    BASE_CSV      = os.path.join(RAW_DIR, "scoring_population.csv")          # built from root below (not Drive)
    MAPPING_CSV   = os.path.join(RAW_DIR, "final_ids_opp_lead_mapping.csv")  # built from root below
    OPP_LOAN_CSV  = os.path.join(RAW_DIR, CFG["opploan"])
    CASE_DATA_CSV = os.path.join(RAW_DIR, CFG["case"])
else:
    BASE_CSV      = os.path.join(DRIVE_ROOT, "data", "df_complete_base_with_partner_filtered.csv")
    MAPPING_CSV   = DRIVE_MAPPING_CSV
    OPP_LOAN_CSV  = os.path.join(QF_DATA_DIR, CFG["opploan"])
    CASE_DATA_CSV = os.path.join(QF_DATA_DIR, CFG["case"])
INQUIRY_CSV = os.path.join((os.path.join(RAW_DIR, CFG["scrub"]) if BUILD_FROM_ROOT
                            else os.path.join(DRIVE_ROOT, "scrub", CFG["scrub"])), "Inquiry.csv")
DEMO_CSV   = os.path.join(RAW_DIR, f"demo_{COHORT}.csv")
FEATURES_OUT    = os.path.join(OUT_DIR, f"query_model_features_{COHORT}.csv")
PREDICTIONS_OUT = os.path.join(OUT_DIR, f"query_predictions_{COHORT}.csv")

print("COHORT          :", COHORT, "| year_month:", YEAR_MONTH, "| snapshot:", SNAPSHOT_DATE)
print("model           :", MODEL_QUERY_PKL)
print("BUILD_FROM_ROOT :", BUILD_FROM_ROOT, "| case data:", CASE_DATA_CSV)

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import json, gc
from tqdm import tqdm
pd.set_option('display.max_columns', None)

## 0b. Environment check (run first)
Warn-only: the trained models were serialized with **xgboost 1.7.4** and the 1.7.4 wheel targets the numpy 1.x ABI (so `numpy<2.0`). See `requirements.txt`.

In [ ]:
# ============================ Environment check (run me first) ============================
# Production-matched runtime (per handover): xgboost 2.1.1, numpy 2.2.6, pandas 2.3.0 on Python 3.11/3.12.
# NOTE: the models were TRAINED/serialized with xgboost 1.7.4 but production LOADS & scores them with
# 2.1.1 (XGBoost loads older model formats; prediction is deterministic). Warn-only (non-fatal).
import sys as _sys_envchk, numpy as _np_envchk
try:
    import xgboost as xgb
except Exception as _e:
    xgb = None; print('[env-check] WARNING: xgboost not importable:', _e)
_EXPECTED_XGB = '2.1.1'
if _sys_envchk.version_info[:2] >= (3, 13):
    print(f"[env-check] WARNING: Python {_sys_envchk.version_info.major}.{_sys_envchk.version_info.minor} "
          f"- production runs 3.11/3.12. pandas groupby.cumcount() crashes on 3.14 (on-us step). "
          f"Use a Python 3.11/3.12 kernel (see requirements.txt).")
if xgb is not None and xgb.__version__ != _EXPECTED_XGB:
    print(f"[env-check] WARNING: xgboost {xgb.__version__} != production {_EXPECTED_XGB}. "
          f"Install the pinned set (pip install -r requirements.txt) to match production.")
elif xgb is not None:
    print(f"[env-check] OK: xgboost {xgb.__version__}")
if _np_envchk.__version__ != '2.2.6':
    print(f"[env-check] NOTE: numpy {_np_envchk.__version__} != production 2.2.6.")


## 1b. Case leakage guard (fatal) + bureau vintage check (advisory / warn-only)
> The query model uses only dim_cases + demographics, so the CRIF bureau-vintage check is
> advisory here (warn, do not fail). The strict `createddate < SNAPSHOT_DATE` case cutoff is the real guard.

In [ ]:
def assert_no_future_dates(df, date_col, source_name, snapshot_date, when="event"):
    snap = pd.Timestamp(snapshot_date).normalize()
    if df is None or len(df) == 0 or date_col not in getattr(df, "columns", []):
        print(f"[leakage-check] {source_name}: '{date_col}' - no rows (ok)"); return
    col = pd.to_datetime(df[date_col], errors="coerce")
    try: col = col.dt.tz_localize(None)
    except (TypeError, AttributeError): pass
    mx = col.max()
    if pd.isna(mx): print(f"[leakage-check] {source_name}: '{date_col}' - all null (ok)"); return
    if mx.normalize() > snap:
        n=int((col.dt.normalize()>snap).sum())
        raise AssertionError(f"FUTURE-DATA LEAKAGE in {source_name}: '{date_col}' has {n} row(s) after snapshot. "
                             f"max={mx.date()} > SNAPSHOT_DATE={snap.date()}. Only {when} data on/before snapshot allowed.")
    print(f"[leakage-check] OK - {source_name}: max({date_col})={mx.date()} <= SNAPSHOT_DATE={snap.date()}")

def assert_bureau_vintage(inquiry, snapshot_date, source_name="CRIF Inquiry scrub", retro_col="RETRO-PR-DT"):
    if isinstance(inquiry, str):
        if not os.path.exists(inquiry): print(f"[vintage-check] {source_name}: file not found - skipped"); return
        try: df = pd.read_csv(inquiry, sep="|", usecols=[retro_col])
        except (ValueError, KeyError): print(f"[vintage-check] {source_name}: no '{retro_col}' - skipped"); return
    else: df = inquiry
    if df is None or len(df)==0 or retro_col not in getattr(df,"columns",[]):
        print(f"[vintage-check] {source_name}: no '{retro_col}' data - skipped"); return
    snap = pd.Timestamp(snapshot_date).normalize()
    parsed = pd.to_datetime(df[retro_col].astype(str).str.strip(), dayfirst=True, errors="coerce").dt.normalize()
    found = sorted(pd.Timestamp(d) for d in parsed.dropna().unique())
    if not found: print(f"[vintage-check] {source_name}: all null - skipped"); return
    bad=[d for d in found if d!=snap]
    if bad:
        # NON-FATAL in this query notebook: model_cust_query is built only from dim_cases
        # (createddate < SNAPSHOT_DATE) + demographics and never consumes the CRIF scrub.
        # The handover query flow (2_6_oot_data_prep / 2_9_oot_validation) has no bureau-vintage
        # check at all, so we WARN instead of failing to avoid coupling query scoring to an
        # unused bureau artifact (e.g. a quarterly dump whose retro date is not the month-end).
        print(f"[vintage-check] WARNING - {source_name}: RETRO-PR-DT {[d.date() for d in found]} != "
              f"SNAPSHOT_DATE={snap.date()} (query model does not use the bureau scrub; continuing)")
        return
    print(f"[vintage-check] OK - {source_name}: RETRO-PR-DT == SNAPSHOT_DATE ({snap.date()})")

## 2. Redshift connection (env-var creds only)

In [ ]:
# --- load .env (robust): strips stray quotes / spaces / trailing commas so a malformed
#     value like  DB_HOST = 'host',  can't break psycopg2. .env is the source of truth. ---
def _load_dotenv(path=None):
    path = path or os.path.join(os.getcwd(), ".env")
    if not os.path.exists(path):
        print("[env] no .env found at", path); return
    for _line in open(path):
        _line = _line.strip()
        if not _line or _line.startswith("#") or "=" not in _line:
            continue
        _k, _v = _line.split("=", 1)
        _k = _k.strip()
        _v = _v.strip().strip(",").strip().strip('\"').strip("'").strip()
        if _k:
            os.environ[_k] = _v
    print("[env] loaded .env ->", path)
_load_dotenv()

def get_redshift_conn():
    try:
        import psycopg2
        return psycopg2.connect(dbname=os.getenv("DB_NAME","dwh_dmi"), user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"), host=os.getenv("DB_HOST","dmi-prod.cqmjwdcqo0bv.ap-south-1.redshift.amazonaws.com"),
            port=os.getenv("DB_PORT","5439"))
    except Exception as e:
        print("Redshift connection unavailable:", e); return None
def split(list_a, chunk_size):
    for i in tqdm(range(0, len(list_a), chunk_size)):
        yield list_a[i:i+chunk_size]

In [ ]:
# fifo (verbatim) — needed to derive payment_number/remaining_payments for the scoring population
def fifo(df_opp_path, df_payment_path, date):
    df_opp = pd.read_csv(df_opp_path)
    df_payments = pd.read_csv(df_payment_path)
    current_date = pd.Timestamp(date)

    previous_month_date = current_date.replace(day=20) - relativedelta(months=1)
    previous_month_timestamp = pd.Timestamp(previous_month_date)
    current_date_timestamp = pd.Timestamp(current_date)

    # fill payment_received_date__c null values and create cols list

    today_str = current_date.strftime('%Y-%m-%d')

    # to select ids whoes closedate was before previous_month_date
    print('previous_month_date',previous_month_date)

    # select all-due dates before current_date
    print('current_date_timestamp',current_date_timestamp)

    df_opp['createddate_opp'] = pd.to_datetime(df_opp['createddate_opp'])
    df_opp['closedate'] = pd.to_datetime(df_opp['closedate'])
    df_opp = df_opp[~df_opp['stagename'].isin(['Canceled', 'Rejected', 'Disbursement Approved', 'Restructured', 'InitialApproach', 'Credit Approved'])]
    df_opp['maturity_date__c'] = pd.to_datetime(df_opp['maturity_date__c'],format='%Y-%m-%d')
    df_opp['actual_maturity_date__c'] = pd.to_datetime(df_opp['actual_maturity_date__c'],format='%Y-%m-%d')
    df_opp['closedate'] = pd.to_datetime(df_opp['closedate'],format='%Y-%m-%d')
    df_opp = df_opp[df_opp['closedate'] <= previous_month_timestamp]


    df_opps_payments = pd.merge(df_payments, df_opp, left_on = 'opportunity__c', right_on = 'opp_id', how = 'inner')

    df_opps_payments['date__c'] = pd.to_datetime(df_opps_payments['date__c'])

    df_opps_payments.sort_values(by = ['opp_id', 'date__c'], inplace = True)

    df_opps_payments['closedate'] = pd.to_datetime(df_opps_payments['closedate'])

    df_opps_payments = df_opps_payments[df_opps_payments['date__c'] < current_date_timestamp]

    print('max_date__c',df_opps_payments['date__c'].max())
    print('Maximum due date(date__c) should not more than today date')


    df_opps_payments['payment_received_date__c'] = pd.to_datetime(df_opps_payments['payment_received_date__c'])

    df_opps_payments.sort_values(by = ['opp_id', 'date__c'], inplace = True)

    df_opps_payments.reset_index(drop = True, inplace = True)

    df_opps_payments['payment_received_date__c'] = np.where((df_opps_payments['stagename'].isin(['Matured', 'Matured Closed'])) & (df_opps_payments['payment_received_date__c'].isnull()),df_opps_payments.actual_maturity_date__c, df_opps_payments.payment_received_date__c)
    df_opps_payments['payment_received_date__c'].fillna(pd.to_datetime(today_str), inplace = True)

    ps_data_date__c = df_opps_payments.drop(['payment_received_date__c'], axis = 1)
    ps_data_prd = df_opps_payments[['opportunity__c','payment_received_date__c']]

    #sorting each dataframe
    ps_data_prd.sort_values(by = ['opportunity__c','payment_received_date__c'],inplace=True)
    ps_data_date__c.sort_values(['opportunity__c','date__c'],inplace=True)


    ps_data_prd.reset_index(drop=True, inplace=True)
    ps_data_date__c.reset_index(drop=True, inplace=True)
    df_opps_payments = pd.concat([ps_data_date__c,ps_data_prd],ignore_index=False,axis=1)
    df_opps_payments_2 = df_opps_payments[['opp_id', 'date__c',
                                       'payment_received_date__c',
                                       'actual_maturity_date__c','loan_tenor_in_month__c',
                                       'sanction_amount__c','net_payment__c','closingprincipalbal__c', 'stagename']].copy(deep = True)


    df_opps_payments_2.reset_index(drop = True, inplace  = True)
    df_opps_payments_final = df_opps_payments_2.copy(deep = True)
    df_opps_payments_final.sort_values(by = ['opp_id', 'date__c'], inplace  = True)
    df_opps_payments_final_2 = df_opps_payments_final.copy(deep = True)
    df_opps_payments_final_2['delinquency'] = (df_opps_payments_final_2['payment_received_date__c'] - df_opps_payments_final_2['date__c']).dt.days
    df_opps_payments_final_w_date = pd.merge(df_opps_payments_final_2, df_opps_payments[['opp_id', 'closedate']].drop_duplicates(), on = 'opp_id', how = 'inner')
    df_opps_payments_final_w_date.sort_values(['opp_id','date__c'], ascending=True,inplace=True)
    df_opps_payments_final_w_date['payment_number']=df_opps_payments_final_w_date.groupby(['opp_id'])['date__c'].cumcount()+1
    df_opps_payments_final_w_date['closedate'] = pd.to_datetime(df_opps_payments_final_w_date['closedate'])
    df_opps_payments_final_w_date['closedate'] = pd.to_datetime(df_opps_payments_final_w_date['closedate'],utc=True)

    df_opps_payments_final_w_date['payment_received_date__c'] = pd.to_datetime(df_opps_payments_final_w_date['payment_received_date__c'])

    df_x =df_opps_payments_final_w_date[df_opps_payments_final_w_date['payment_received_date__c']>today_str]
    df_y =df_opps_payments_final_w_date[~(df_opps_payments_final_w_date['payment_received_date__c']>today_str)]
    df_x['payment_received_date__c']= today_str
    df_opps_payments_final_w_date = pd.concat([df_x,df_y])
    df_opps_payments_final_w_date['payment_received_date__c'] = pd.to_datetime(df_opps_payments_final_w_date['payment_received_date__c'])
    df_opps_payments_final_w_date['date__c'] = pd.to_datetime(df_opps_payments_final_w_date['date__c'])

    df_opps_payments_final_w_date['delinquency'] = (df_opps_payments_final_w_date['payment_received_date__c'] - df_opps_payments_final_w_date['date__c']).dt.days
    df_opps_payments_final_w_date.reset_index(drop = True, inplace = True)
    return df_opps_payments_final_w_date

## Stage 0 (shared root) — build base + mapping from Redshift (no Drive dependency)

In [ ]:
# ============================ Stage 0 (shared root): build base + mapping from Redshift ============================
# Builds the SAME scoring population + lead<->loan mapping that feat_crn builds — locally, NOT from Drive.
# Idempotent: if feat_crn already produced them in this cohort's RAW_DIR, reuse them.
def build_scoring_base_and_mapping():
    if os.path.exists(BASE_CSV) and os.path.exists(MAPPING_CSV):
        print("base + mapping already present locally, reusing:"); print("  ", BASE_CSV); print("  ", MAPPING_CSV); return
    conn = get_redshift_conn(); assert conn is not None, "Set DB_* env vars for BUILD_FROM_ROOT=True."
    # 0-A: schedule + loan master + payments -> FIFO
    q_sched = f"""select ps.loan_id as opportunity__c, o.loan_number, o.loantenor_in_month,
                         o.partner_name, o.created_date, ps.ps_id as id_pay_sch,
                         ps.created_date as createddate_paysch, ps.emi_due_date as date__c,
                         CAST(ps.pmt_rec_date AS VARCHAR) as payment_received_date__c
                  from lms.fact_repayment_schedule as ps join lms.dim_loan o on o.loan_id = ps.loan_id
                  where ps.emi_due_date BETWEEN '{BASE_DATE_START}' AND '{BASE_DATE_END}'
                    and (ps.ps_id not in (select distinct record_id from lms.dim_purge_records where object_name='Payment Schedule'))
                    and o.product_type = 'Personal Loan'"""
    df_sched = pd.read_sql(q_sched, conn)
    df_ids = df_sched.drop_duplicates(subset='opportunity__c', keep='first').rename(columns={'opportunity__c':'opp_id'})
    df_ids.to_csv(os.path.join(RAW_DIR, "opp_ids_between_dates.csv"), index=False)
    opp_ids = df_sched['opportunity__c'].astype(str).unique().tolist()
    df_opps = pd.DataFrame()
    for s in split(opp_ids, 100000):
        q = f"""SELECT O.loan_id as opp_id, O.created_date as createddate_opp, O.stage as stagename,
                       O.maturity_date as maturity_date__c, O.actual_maturity_date as actual_maturity_date__c,
                       O.closed_date as closedate, O.loantenor_in_month as loan_tenor_in_month__c,
                       O.sanction_amount as sanction_amount__c
                FROM dwh_dmi.lms.dim_loan O WHERE O.loan_id in ({','.join([f"'{x}'" for x in s])})"""
        df_opps = pd.concat([pd.read_sql(q, conn), df_opps])
    df_opps.to_csv(os.path.join(RAW_DIR, "opp_data.csv"), index=False)
    payment_df = pd.DataFrame()
    for s in split(opp_ids, 100000):
        q = f"""select ps.ps_id as id_pay_sch, ps.created_date as createddate_paysch, ps.loan_id as opportunity__c,
                       ps.net_pmt as net_payment__c, ps.emi_due_date as date__c, ps.net_payment as netpayment__c,
                       CAST(ps.pmt_rec_date AS VARCHAR) as payment_received_date__c, ps.closingprincipal_bal_pos as closingprincipalbal__c
                FROM dwh_dmi.lms.fact_repayment_schedule as ps where loan_id in ({','.join([f"'{x}'" for x in s])})
                  AND (ps.ps_id not in (select distinct record_id from lms.dim_purge_records where object_name='Payment Schedule'))"""
        payment_df = pd.concat([pd.read_sql(q, conn), payment_df])
    payment_df.to_csv(os.path.join(RAW_DIR, "payment_data.csv"), index=False)
    dfw = fifo(os.path.join(RAW_DIR, "opp_data.csv"), os.path.join(RAW_DIR, "payment_data.csv"), BASE_REF_DATE)  # base-selection as-of (NOT snapshot)
    # 0-B: scoring population (active loans in cohort month) — NO labels
    dfw = dfw.loc[:, ~dfw.columns.str.startswith("Unnamed")]
    dfw['remaining_payments'] = dfw['loan_tenor_in_month__c'] - dfw['payment_number']
    df_ff = dfw[dfw['remaining_payments'] >= MIN_REMAINING_PAYMENTS].copy()
    df_ff['date__c'] = pd.to_datetime(df_ff['date__c'], format='%Y-%m-%d', errors='coerce')
    df_ff = df_ff[df_ff['date__c'] > BASE_MIN_DUE_DATE]
    df_ff['year_month'] = df_ff['date__c'].dt.to_period('M').astype(str)
    part_map = df_ids[['opp_id','partner_name']].drop_duplicates('opp_id')
    popn = pd.merge(df_ff, part_map, on='opp_id', how='left')
    popn = popn[popn['partner_name'].isin(PARTNER_LIST)]
    popn[['opp_id','year_month','partner_name']].drop_duplicates().to_csv(BASE_CSV, index=False)
    # 0-C: parent -> lead mapping
    loan_ids = popn['opp_id'].astype(str).unique().tolist()
    dim_parent = pd.DataFrame()
    for s in split(loan_ids, 100000):
        q = f"""select l.loan_id, l.parent_loan_id from lms.dim_loan l where l.loan_id in ({','.join([f"'{x}'" for x in s])})"""
        dim_parent = pd.concat([pd.read_sql(q, conn), dim_parent])
    parent_ids = dim_parent['parent_loan_id'].dropna().astype(str).unique().tolist()
    dim_lead = pd.DataFrame()
    for s in split(parent_ids, 100000):
        q = f"""select dl.lead_id, l.loan_id, l.parent_loan_id from lms.dim_lead dl
                right join lms.dim_loan l on l.loan_id = dl.loan_id where l.loan_id in ({','.join([f"'{x}'" for x in s])})"""
        dim_lead = pd.concat([pd.read_sql(q, conn), dim_lead])
    conn.close()
    mp = (dim_parent.merge(dim_lead[['loan_id','lead_id']], on='loan_id', how='left')[['lead_id','loan_id']].dropna().drop_duplicates())
    mp.to_csv(MAPPING_CSV, index=False)
    print("built base ->", BASE_CSV, popn['opp_id'].nunique(), "loans | mapping ->", MAPPING_CSV, mp.shape)

if BUILD_FROM_ROOT:
    build_scoring_base_and_mapping()
else:
    print("Shared base/mapping: using existing artifacts (BUILD_FROM_ROOT=False)")

## Stage 0 — cohort pulls (guarded by BUILD_FROM_ROOT). No labels.
0-A opp_id->loan_number (dim_loan); 0-B loan_number->dim_cases (incl. reply_count for v4).

In [ ]:
NEEDED_COLS = ['loan_number','createddate','case_age_in_days','reply_count',
               'qrc','case_origin','closure_remarks','type','sub_type','case_category']
if BUILD_FROM_ROOT:
    conn = get_redshift_conn(); assert conn is not None, "Set DB_* env vars for BUILD_FROM_ROOT=True."
    # cohort scoring population built from root above (BASE_CSV, local — not Drive)
    base = pd.read_csv(BASE_CSV); opp_ids = base[base["year_month"].astype(str)==YEAR_MONTH]["opp_id"].astype(str).unique().tolist()
    opp_df = pd.DataFrame()
    for s in split(opp_ids,100000):
        q=f"""SELECT O.loan_id as opp_id, O.loan_number, O.loan_type FROM dwh_dmi.lms.dim_loan O
              WHERE O.loan_id in ({','.join([f"'{x}'" for x in s])})"""
        opp_df=pd.concat([pd.read_sql(q,conn),opp_df])
    opp_df.to_csv(OPP_LOAN_CSV,index=False)
    loan_numbers = opp_df["loan_number"].dropna().astype(str).unique().tolist()
    cols=", ".join(f"dc.{c}" for c in NEEDED_COLS)
    cases=pd.DataFrame()
    for s in split(loan_numbers,100000):
        q=f"""select {cols} from lms.dim_cases dc where dc.loan_number in ({','.join([f"'{x}'" for x in s])})"""
        cases=pd.concat([cases,pd.read_sql(q,conn)])
    cases.to_csv(CASE_DATA_CSV,index=False); conn.close()
    print("Stage 0 done:", opp_df.shape, cases.shape)
else:
    print("Stage 0 skipped (BUILD_FROM_ROOT=False)")

## 3. Load frozen category map + bureau vintage guard

In [ ]:
with open(CATEGORIES_JSON) as f:
    FINAL_CATEGORIES_MAP = json.load(f)
print("category cols:", list(FINAL_CATEGORIES_MAP.keys()))
# advisory only (warn, non-fatal): query model does not use the bureau scrub
assert_bureau_vintage(INQUIRY_CSV, SNAPSHOT_DATE, source_name=f"CRIF Inquiry scrub [{COHORT}]")

## 4. `create_preclosure_features_v4` — verbatim (full case-feature set)

In [ ]:
import pandas as pd
import numpy as np
import gc
def create_preclosure_features_v4(df_input, snapshot_date_str):
    # --- 1. SETUP & LEAKAGE CONTROL ---
    snap_date = pd.to_datetime(snapshot_date_str)
    print(f"Engineering features for snapshot: {snapshot_date_str}")
    df = df_input.copy()
    df['createddate'] = pd.to_datetime(df['createddate'])
    df = df[df['createddate'] < snap_date]
    globals()['cases_used'] = df   # leakage-guard hook (frame actually consumed)
    for col in FINAL_CATEGORIES_MAP.keys():
        df[col] = df[col].astype(str).replace('Miscellaneous1', 'Miscellaneous')
        df[col] = df[col].where(df[col].isin(FINAL_CATEGORIES_MAP[col]), "Miscellaneous")
    loans = df['loan_number'].unique()
    master_features = pd.DataFrame(index=loans); master_features.index.name = 'loan_number'
    windows = { "7d": 7, "30d": 30, "60d": 60, "90d": 90, "180d": 180 }
    ratio_pairs = [("7d","30d"),("30d","60d"),("30d","90d"),("30d","180d"),("60d","180d"),("90d","180d")]
    # --- 2. NUMERICAL ROOTS ---
    for col in ["case_age_in_days", "reply_count"]:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
        master_features[f'stat_{col}_mean_lth'] = df.groupby('loan_number')[col].mean()
        master_features[f'stat_{col}_max_lth'] = df.groupby('loan_number')[col].max()
        for label, days in windows.items():
            win_df = df[df['createddate'] >= (snap_date - pd.Timedelta(days=days))]
            master_features[f'stat_{col}_mean_{label}'] = win_df.groupby('loan_number')[col].mean()
            master_features[f'stat_{col}_max_{label}'] = win_df.groupby('loan_number')[col].max()
    # --- 3. CATEGORICAL ROOTS ---
    for facet, categories in FINAL_CATEGORIES_MAP.items():
        win_counts = {label: df[df['createddate'] >= (snap_date - pd.Timedelta(days=days))].groupby('loan_number')[facet].count()
                      for label, days in windows.items()}
        for cat in categories:
            clean_cat = f"{facet}_{str(cat).replace(' ', '_').lower()}"
            is_cat_mask = (df[facet] == cat)
            last_date = df[is_cat_mask].groupby('loan_number')['createddate'].max()
            master_features[f'days_since_{clean_cat}'] = (snap_date - last_date).dt.days
            for label, days in windows.items():
                win_mask = df['createddate'] >= (snap_date - pd.Timedelta(days=days))
                vol_col = f"vol_{clean_cat}_{label}"
                master_features[vol_col] = df[win_mask & is_cat_mask].groupby('loan_number')[facet].count()
                master_features[f'mix_{clean_cat}_{label}'] = master_features[vol_col] / win_counts[label].replace(0, 1)
            mask_30 = df['createddate'] >= (snap_date - pd.Timedelta(days=30))
            master_features[f'frict_avg_age_{clean_cat}_30d'] = df[mask_30 & is_cat_mask].groupby('loan_number')['case_age_in_days'].mean()
        gc.collect()
    # --- 4. META FEATURES ---
    mask_60 = df['createddate'] >= (snap_date - pd.Timedelta(days=60)); df_60 = df[mask_60]
    master_features['unique_subtype_count_60d'] = df_60.groupby('loan_number')['sub_type'].nunique()
    master_features['unique_origin_count_60d'] = df_60.groupby('loan_number')['case_origin'].nunique()
    for col in ['sub_type', 'case_category']:
        loop_val = df_60.groupby(['loan_number', col]).size().reset_index(name='c')
        master_features[f'loop_{col}_60d'] = loop_val.groupby('loan_number')['c'].max()
    # --- 5. VELOCITY RATIOS ---
    master_features = master_features.copy()
    for col in ["case_age_in_days", "reply_count"]:
        for w1, w2 in ratio_pairs:
            m1 = master_features[f'stat_{col}_mean_{w1}'].fillna(0); m2 = master_features[f'stat_{col}_mean_{w2}'].fillna(0)
            master_features[f'ratio_{col}_{w1}_{w2}'] = m1.div(m2).fillna(0).replace([np.inf, -np.inf], 0)
    for facet, cats in FINAL_CATEGORIES_MAP.items():
        for cat in cats:
            clean_cat = f"{facet}_{str(cat).replace(' ', '_').lower()}"
            for w1, w2 in ratio_pairs:
                v1 = master_features[f"vol_{clean_cat}_{w1}"].fillna(0); v2 = master_features[f"vol_{clean_cat}_{w2}"].fillna(0)
                master_features[f'vel_{clean_cat}_{w1}_{w2}'] = v1.div(v2).fillna(0).replace([np.inf, -np.inf], 0)
    return master_features.fillna(0).astype('float32')

## 5. Build case features + case leakage guard

In [ ]:
cases = pd.read_csv(CASE_DATA_CSV, usecols=lambda c: c in NEEDED_COLS, low_memory=False)
print("cases:", cases.shape)
case_feats = create_preclosure_features_v4(cases, SNAPSHOT_DATE)
assert_no_future_dates(cases_used, "createddate", "dim_cases (used for case features)", SNAPSHOT_DATE, when="case")
print("case_feats:", case_feats.shape)

## 6. Bridge loan_number -> opp_id -> lead_id

In [ ]:
opp_loan = pd.read_csv(OPP_LOAN_CSV, dtype=str)
opp_lead = pd.read_csv(MAPPING_CSV, dtype=str)
key = case_feats.reset_index()[['loan_number']].drop_duplicates()
key['loan_number'] = key['loan_number'].astype(str)
key = key.merge(opp_loan[['opp_id','loan_number']], on='loan_number', how='left')
id_cols = [c for c in ['loan_id','final_opp_id','parent_loan_id'] if c in opp_lead.columns]
lead_long = pd.concat([opp_lead[['lead_id', c]].rename(columns={c:'opp_id'}) for c in id_cols],
                      ignore_index=True).dropna(subset=['opp_id']).drop_duplicates()
key = key.merge(lead_long, on='opp_id', how='left').drop_duplicates('loan_number')
key['lead_id'] = key['lead_id'].astype(str)
print("matched lead_id:", key['lead_id'].notna().sum(), "/", len(key))

## 7. Demographics (`net_monthly_sal`, `age_bucket`, `mailing_state`) — from dim_lead / dim_lead_contact

In [ ]:
if os.path.exists(DEMO_CSV):
    df_demo = pd.read_csv(DEMO_CSV)
else:
    conn = get_redshift_conn()
    if conn is None:
        print("No Redshift + no cached demo; demographics -> NaN")
        df_demo = pd.DataFrame(columns=['lead_id'] + ADDITIONAL_FEATS)
    else:
        lead_list = key['lead_id'].dropna().astype(str).unique().tolist()
        df_demo = pd.DataFrame()
        for s in split(lead_list, 100000):
            q = f"""select l.lead_id, l.net_monthly_sal, c.age_bucket, c.mailing_state
                    from lms.dim_lead l left join lms.dim_lead_contact c on l.lead_id = c.lead_id
                    where l.lead_id in ({','.join([f"'{x}'" for x in s])})"""
            df_demo = pd.concat([df_demo, pd.read_sql(q, conn)])
        conn.close(); df_demo = df_demo.drop_duplicates(); df_demo.to_csv(DEMO_CSV, index=False)
print("df_demo:", df_demo.shape)

## 8. Assemble model-ready frame (case + demographics), keyed by lead_id (`multibureau__c`)

In [ ]:
df_demo['lead_id'] = df_demo['lead_id'].astype(str)
key_demo = key.merge(df_demo, on='lead_id', how='left')
feat = case_feats.reset_index().merge(
    key_demo[['loan_number','lead_id'] + ADDITIONAL_FEATS].drop_duplicates('loan_number'),
    on='loan_number', how='left')
# Match the handover (2_6_oot_data_prep): key by lead_id (upper), keep per-loan rows (no
# multibureau__c dedup — case features are loan-level, and the base is ~1 loan per lead).
feat['multibureau__c'] = feat['lead_id'].astype(str).str.upper()
# drop rows with no lead mapping (handover dropped them via inner merge); lead_id may be the
# string 'nan'/'none' after an earlier astype(str), so exclude those explicitly.
feat = feat[~feat['multibureau__c'].isin(['NAN', 'NONE', ''])]
_dups = int(feat['multibureau__c'].duplicated().sum())
if _dups:
    print(f"NOTE: {_dups} lead(s) have >1 loan with cases (multi-loan lead) -> kept per-loan rows, "
          f"matching the handover (the ensemble's duplicate-key guard will flag these if present).")
print("assembled:", feat.shape)

## 9. Score with the trained query model (`model_cust_query.pkl`)
Select exactly `model.feature_names`; predict; save. No labels/fitting.

In [ ]:
import xgboost as xgb, pickle

with open(MODEL_QUERY_PKL, "rb") as _fh:
    model = pickle.load(_fh)
model_features = list(model.feature_names)
print("model expects", len(model_features), "features")
missing = [c for c in model_features if c not in feat.columns]
assert not missing, f"Pipeline did not produce {len(missing)} model feature(s): {missing[:20]}{' ...' if len(missing)>20 else ''}"
# Match 2_9 (handover) exactly: select model features, inf->nan, DMatrix, predict.
# No categorical handling / encoding — the handover fed these straight to DMatrix; the model's
# demographic features (net_monthly_sal, age_bucket) are numeric. (mailing_state is a raw string
# in the data and is NOT a trained model feature, else the handover's plain DMatrix would error.)
X = feat.set_index("multibureau__c")[model_features].replace([np.inf, -np.inf], np.nan)
X.to_csv(FEATURES_OUT)
dmat = xgb.DMatrix(X[model_features], feature_names=model_features)
scores = model.predict(dmat)
predictions = pd.DataFrame({"multibureau__c": X.index, "preclosure_score_query": scores})
predictions.to_csv(PREDICTIONS_OUT, index=False)
print("saved features ->", FEATURES_OUT, X.shape)
print("saved preds    ->", PREDICTIONS_OUT, predictions.shape)
predictions.head()

## 10. Prediction EDA

In [ ]:
import matplotlib.pyplot as plt
print(predictions["preclosure_score_query"].describe())
plt.figure(figsize=(7,4)); plt.hist(predictions["preclosure_score_query"], bins=50)
plt.title(f"Query-model pre-closure score — {COHORT}"); plt.tight_layout(); plt.show()

---
**Scoring only.** No `preclosure_flag`, no 90-day outcome, no split, no fitting. Case cutoff is
strict `createddate < SNAPSHOT_DATE`. Output: `query_model_features_<cohort>.csv` + `query_predictions_<cohort>.csv`.